# Nhận diện ngôn ngữ ký hiệu qua webcam

**Sổ Nghiên Cứu AI, dự án mẫu 4/10.** Trích điểm mốc bàn tay bằng MediaPipe rồi so sánh mô hình một khung hình với mô hình chuỗi.

**Câu hỏi nghiên cứu:** Mô hình học chuỗi trên điểm mốc bàn tay có nhận diện ký hiệu động tốt hơn mô hình chỉ nhìn một khung hình không?

**Giả thuyết:** LSTM trên chuỗi 30 khung đạt độ chính xác cao hơn MLP một khung với các ký hiệu có chuyển động, kể cả trên người ký chưa gặp.

### Cách dùng notebook này
1. Trên Colab, chọn **Runtime → Change runtime type → T4 GPU** nếu có (không bắt buộc).
2. Chạy lần lượt từng ô bằng **Shift + Enter**. Lần đầu có thể mất vài phút để cài thư viện và tải dữ liệu.
3. Các ô **Việc của em** là chỗ em tự làm tiếp để biến dự án mẫu thành nghiên cứu của riêng mình.

Phần 1 trích điểm mốc từ video của em. Phần 2 chạy ngay với **dữ liệu điểm mốc giả lập** để em hiểu vì sao cần mô hình chuỗi; khi có video thật, phần 2 tự dùng dữ liệu của em.

> **Liêm chính khoa học.** Đây là mã mẫu để luyện tập. Nếu dùng cho dự án dự thi, hãy ghi rõ trong sổ nghiên cứu và báo cáo đoạn mã nào lấy từ notebook này, đoạn nào em tự viết hoặc sửa. Dữ liệu dự thi phải do em thu thập hoặc lấy từ nguồn công khai có ghi nguồn.

In [ ]:
%pip install -q mediapipe

In [ ]:
import os, random, time, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED); np.random.seed(SEED)
plt.rcParams["figure.dpi"] = 110
def tieu_de(s): print("\n" + "=" * 8, s, "=" * 8)

import torch
from torch import nn
_ = torch.manual_seed(SEED)
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay
SO_KHUNG, SO_DIEM = 30, 21 * 3          # 30 khung, 21 điểm mốc x (x, y, z)

## Phần 1. Trích điểm mốc từ video của em
Xếp video theo `videos/<tu_ky_hieu>/<nguoi>_<lan>.mp4`, ví dụ `videos/cam_on/an_01.mp4`. Tên người ở đầu tên file để chia dữ liệu theo người. **Chỉ quay khi có giấy đồng ý.**

In [ ]:
import cv2, urllib.request
def trich_video(duong_dan, detector):
    import mediapipe as mp
    cap = cv2.VideoCapture(duong_dan); fps = cap.get(cv2.CAP_PROP_FPS) or 30; khung = []; i = 0
    while True:
        ok, f = cap.read()
        if not ok: break
        anh = mp.Image(image_format=mp.ImageFormat.SRGB, data=cv2.cvtColor(f, cv2.COLOR_BGR2RGB))
        r = detector.detect_for_video(anh, int(i * 1000 / fps)); i += 1
        if r.hand_landmarks:
            p = np.array([[d.x, d.y, d.z] for d in r.hand_landmarks[0]])
            p = p - p[0]; p = p / (np.abs(p).max() + 1e-6)          # chuẩn hóa: gốc ở cổ tay, cùng tỉ lệ
            khung.append(p.ravel())
    cap.release()
    if len(khung) < 5: return None
    idx = np.linspace(0, len(khung) - 1, SO_KHUNG).astype(int)       # lấy đều 30 khung
    return np.array(khung)[idx]

X_THAT = None
if os.path.isdir("videos"):
    import mediapipe as mp
    from mediapipe.tasks import python as mpt
    from mediapipe.tasks.python import vision
    urllib.request.urlretrieve("https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task", "hand.task")
    TU = sorted(os.listdir("videos")); X, Y, NG = [], [], []
    for yi, tu in enumerate(TU):
        for f in sorted(os.listdir(f"videos/{tu}")):
            det = vision.HandLandmarker.create_from_options(vision.HandLandmarkerOptions(base_options=mpt.BaseOptions(model_asset_path="hand.task"), running_mode=vision.RunningMode.VIDEO, num_hands=1))
            s = trich_video(f"videos/{tu}/{f}", det); det.close()
            if s is not None: X.append(s); Y.append(yi); NG.append(f.split("_")[0])
    X_THAT, Y_THAT, NG_THAT = np.array(X), np.array(Y), np.array(NG)
    print("Đã trích", len(X), "video,", len(TU), "từ,", len(set(NG)), "người")
else:
    print("Chưa có thư mục videos/. Phần 2 sẽ dùng dữ liệu giả lập.")

## Phần 2. Vì sao cần mô hình chuỗi?
Dữ liệu giả lập có 6 ký hiệu. Ba cặp ký hiệu **kết thúc ở cùng một tư thế tay** nhưng khác nhau ở chuyển động. Mô hình chỉ nhìn khung cuối sẽ nhầm các cặp này.

In [ ]:
def gia_lap(so_nguoi=8, lap=12):
    rng = np.random.default_rng(SEED); tu_the = rng.normal(0, 1, (3, SO_DIEM)); huong = rng.normal(0, 1, (6, SO_DIEM))
    X, Y, NG = [], [], []
    for ng in range(so_nguoi):
        lech = rng.normal(0, .25, SO_DIEM)                              # mỗi người ký hơi khác
        for k in range(6):
            dich = tu_the[k // 2]                                       # 2 ký hiệu chung một tư thế cuối
            for _ in range(lap):
                t = np.linspace(1, 0, SO_KHUNG)[:, None]
                X.append(dich + lech + t * huong[k] * 1.5 + rng.normal(0, .3, (SO_KHUNG, SO_DIEM))); Y.append(k); NG.append(f"nguoi{ng}")
    return np.array(X, dtype=np.float32), np.array(Y), np.array(NG)
if X_THAT is not None: X, Y, NG = X_THAT.astype(np.float32), Y_THAT, NG_THAT; NGUON = "video của em"
else: X, Y, NG = gia_lap(); NGUON = "GIẢ LẬP"
nguoi = sorted(set(NG)); kiem_tra = nguoi[-max(1, len(nguoi) // 4):]            # người ký CHƯA GẶP
mtr, mte = ~np.isin(NG, kiem_tra), np.isin(NG, kiem_tra)
print(NGUON, "| huấn luyện", mtr.sum(), "mẫu | kiểm tra", mte.sum(), "mẫu của", kiem_tra)

In [ ]:
EPOCHS = int(os.environ.get("SNC_EPOCHS", 40))
class MLP(nn.Module):
    def __init__(s, k): super().__init__(); s.f = nn.Sequential(nn.Linear(SO_DIEM, 128), nn.ReLU(), nn.Dropout(.2), nn.Linear(128, k))
    def forward(s, x): return s.f(x[:, -1])                              # chỉ nhìn khung cuối
class LSTM(nn.Module):
    def __init__(s, k): super().__init__(); s.l = nn.LSTM(SO_DIEM, 64, batch_first=True); s.o = nn.Linear(64, k)
    def forward(s, x): return s.o(s.l(x)[0][:, -1])
def train_eval(cls):
    k = len(set(Y)); m = cls(k); opt = torch.optim.Adam(m.parameters(), 1e-3)
    xt, yt = torch.tensor(X[mtr]), torch.tensor(Y[mtr])
    for ep in range(EPOCHS):
        m.train(); perm = torch.randperm(len(xt))
        for i in range(0, len(xt), 32):
            b = perm[i:i + 32]; opt.zero_grad(); nn.functional.cross_entropy(m(xt[b]), yt[b]).backward(); opt.step()
    m.eval(); p = m(torch.tensor(X[mte])).argmax(1).numpy(); return p
KQ = {}
for ten, cls in {"MLP một khung (đối chứng)": MLP, "LSTM chuỗi 30 khung": LSTM}.items():
    p = train_eval(cls); KQ[ten] = accuracy_score(Y[mte], p); print(f"{ten}: độ chính xác trên người chưa gặp = {KQ[ten]:.3f}")
    ConfusionMatrixDisplay(confusion_matrix(Y[mte], p)).plot(cmap="Blues"); plt.title(ten); plt.show()

## Việc của em
1. Hợp tác với trường hoặc trung tâm hỗ trợ người khiếm thính để chọn 15 đến 30 từ thông dụng, nhờ người dùng ngôn ngữ ký hiệu thật hướng dẫn.
2. Quay mỗi từ ít nhất 10 lần với 5 người trở lên. Giữ cách đặt tên file để chia theo người.
3. Báo cáo độ chính xác trên **người chưa gặp**, vì đó mới là điều kiện dùng thật.
4. Đo độ trễ khi chạy trực tiếp trên webcam laptop.